# 🌦️ Weather Forecast Pipeline – Trembling Process

> **Mục đích:** Tổng hợp toàn bộ các lệnh chạy tuần tự của hệ thống dự báo thời tiết – từ **crawl dữ liệu** đến **dự báo (forecast)**.  
> Chỉ cần chạy lần lượt các cell bên dưới là xong.

---

## 📊 Sơ đồ luồng xử lí

```mermaid
graph TD

  START(("🌦️"))
  START --> A1
  START --> A2
  START --> A3

  A1["🌐 Vrain API"]
  A2["🖥️ Selenium"]
  A3["☁️ OpenWeather"]

  A1 --> RAW
  A2 --> RAW
  A3 --> RAW

  RAW[/"📁 data∕data_crawl — files thô"\]

  RAW --> MERGE

  MERGE["📦 MERGE — Gộp 40+ cột"]

  MERGE --> MERGED
  MERGED[/"📁 merged_vrain_data.xlsx"\]

  MERGED --> CLEARDATA

  CLEARDATA["🧹 CLEARDATA — Rename EN ∙ Fix dtype ∙ Fill NaN"]

  CLEARDATA --> CLEANED
  CLEANED[/"📁 *_cleaned_*.csv"\]

  CLEANED --> TARGET

  TARGET["🎯 CLEAN TARGET — Loại NaN∕inf rain_total"]

  TARGET --> FEATURES

  FEATURES["🔍 CLEAN FEATURES — Loại NaN∕inf∕outlier"]

  FEATURES --> READY
  READY[/"📁 *.clean_final.csv"\]

  READY --> CONFIG

  CONFIG["📋 LOAD CONFIG + FILTER BAD ROWS"]

  CONFIG --> SPLITD

  SPLITD["✂️ SPLIT — 80∕10∕10 time-series"]

  SPLITD --> FEAT

  FEAT["🔧 FEATURES — Lag ∙ Rolling ∙ Time ∙ Location ∙ Interact ∙ Diff"]

  FEAT --> SELEC

  SELEC["🗑️ SELECT — SHAP top-k ≤ 50"]

  SELEC --> TRANSFORM

  TRANSFORM["🔄 TRANSFORM — Impute ∙ Scale ∙ Encode ∙ Polynomial"]

  TRANSFORM --> TRAIN

  TRAIN["🤖 TRAIN — XGB ∙ LGBM ∙ CatBoost ∙ RF ∙ TwoStage"]

  TRAIN --> EVALUATE

  EVALUATE["📊 EVALUATE — R² · RMSE · MAE · MBE · Pearson · CSI · F1"]

  EVALUATE --> ARTIFACTS
  ARTIFACTS[/"📁 ML_artifacts∕latest — Model.pkl"\]

  ARTIFACTS --> PREDICT
  ARTIFACTS --> DIAG

  PREDICT["⚡ PREDICT — Load ∙ Build ∙ Transform ∙ Output"]

  PREDICT --> DONE
  DONE(("✅"))

  DIAG["🔍 DIAGNOSTICS — Top-50 errors ∙ Station RMSE"]

  DIAG -.-> CONFIG

  OPTUNA["🔬 OPTUNA — TPE 100+ trials"]
  OPTUNA -.-> CONFIG

  style START fill:#263238,stroke:#263238,color:#fff
  style DONE fill:#2e7d32,stroke:#2e7d32,color:#fff

  style A1 fill:#e3f2fd,stroke:#1976d2,color:#0d47a1
  style A2 fill:#e3f2fd,stroke:#1976d2,color:#0d47a1
  style A3 fill:#e3f2fd,stroke:#1976d2,color:#0d47a1

  style RAW fill:#f5f5f5,stroke:#9e9e9e,color:#424242
  style MERGED fill:#f5f5f5,stroke:#9e9e9e,color:#424242
  style CLEANED fill:#f5f5f5,stroke:#9e9e9e,color:#424242
  style READY fill:#f5f5f5,stroke:#9e9e9e,color:#424242
  style ARTIFACTS fill:#f5f5f5,stroke:#9e9e9e,color:#424242

  style MERGE fill:#fff3e0,stroke:#ef6c00,color:#e65100
  style CLEARDATA fill:#fff3e0,stroke:#ef6c00,color:#e65100

  style TARGET fill:#e8f5e9,stroke:#388e3c,color:#1b5e20
  style FEATURES fill:#e8f5e9,stroke:#388e3c,color:#1b5e20

  style CONFIG fill:#fce4ec,stroke:#d81b60,color:#880e4f
  style SPLITD fill:#fce4ec,stroke:#d81b60,color:#880e4f
  style FEAT fill:#fce4ec,stroke:#d81b60,color:#880e4f
  style SELEC fill:#fce4ec,stroke:#d81b60,color:#880e4f
  style TRANSFORM fill:#fce4ec,stroke:#d81b60,color:#880e4f
  style TRAIN fill:#fce4ec,stroke:#d81b60,color:#880e4f
  style EVALUATE fill:#fce4ec,stroke:#d81b60,color:#880e4f

  style PREDICT fill:#e8f5e9,stroke:#2e7d32,color:#1b5e20

  style DIAG fill:#e0f7fa,stroke:#00838f,color:#006064
  style OPTUNA fill:#f3e5f5,stroke:#8e24aa,color:#4a148c
```

| Màu | Ý nghĩa |
|-----|---------|
| 🔵 Xanh dương | Thu thập dữ liệu |
| 🟠 Cam | Gộp & làm sạch sơ bộ |
| 🟢 Xanh lá | Làm sạch target & features |
| 🔴 Hồng | Huấn luyện mô hình |
| 🟣 Tím | Optuna tuning (optional) |
| 🩵 Cyan | Diagnostics & feedback |
| ⬜ Xám | Nơi lưu dữ liệu |
| ‑ ‑ → | Luồng optional / feedback |

---

## ⚙️ Setup ban đầu

Cell này chuyển về **thư mục gốc của project** để tất cả các lệnh bên dưới chạy đúng đường dẫn.  
Nếu bạn đã ở thư mục gốc rồi thì không cần chạy cell này.

In [49]:
import os, sys, subprocess

# Tim thu muc goc project (chua manage.py)
# Notebook o: .../PROJECT_WEATHER_FORCAST/Weather_Forcast_App/Evaluate_accuracy/
# Project root: .../PROJECT_WEATHER_FORCAST/
_candidates = [
    r"d:\PROJECT_WEATHER_FORCAST",
    os.path.abspath(os.path.join(os.getcwd(), "..", "..")),
    os.path.abspath(os.path.join(os.getcwd(), "..")),
    os.getcwd(),
]
PROJECT_ROOT = None
for _c in _candidates:
    if os.path.isfile(os.path.join(_c, "manage.py")):
        PROJECT_ROOT = _c
        break

if PROJECT_ROOT is None:
    raise RuntimeError("Khong tim thay manage.py! Sua duong dan trong o dau tien.")

os.chdir(PROJECT_ROOT)
print(f"Working directory: {os.getcwd()}")

def run_script(script, *args):
    """Chay script Python tu thu muc goc project"""
    cmd = [sys.executable, script, *args]
    print(f"▶ {' '.join(cmd)}")
    r = subprocess.run(cmd, cwd=PROJECT_ROOT)
    if r.returncode != 0:
        print(f"⚠️ Exit code: {r.returncode}")
    return r.returncode

Working directory: d:\PROJECT_WEATHER_FORCAST


---

## 🕷️ Phase 1 – Thu thập dữ liệu (Data Crawling)

Hệ thống hỗ trợ **3 nguồn crawl** dữ liệu thời tiết:

| Nguồn | Script | Mô tả |
|--------|--------|---------|
| **Vrain API** | `Crawl_data_from_Vrain_by_API.py` | Nhanh nhất – gọi API của vrain.vn, multi-threaded |
| **Vrain Selenium** | `Crawl_data_from_Vrain_by_Selenium.py` | Dùng Selenium mở trình duyệt headless để crawl |
| **OpenWeather + WeatherAPI** | `Crawl_data_by_API.py` | Gọi API của OpenWeather và WeatherAPI |

**Output:** Các file `.xlsx` / `.csv` được lưu vào `data/data_crawl/`.

**Lưu ý:**  
- Cần có file `.env` chứa các API key (`OPENWEATHER_API_KEY`, `WEATHERAPI_KEY`)  
- Vrain Selenium cần cài Chrome + ChromeDriver  
- Chỉ cần chạy **1 trong 3** nguồn tùy theo nhu cầu, hoặc chạy hết để gộp nhiều nguồn

In [50]:
# ====== CACH 1: Crawl tu Vrain API (KHUYEN DUNG - nhanh nhat) ======
run_script("Weather_Forcast_App/scripts/Crawl_data_from_Vrain_by_API.py")

▶ d:\PROJECT_WEATHER_FORCAST\.venv\Scripts\python.exe Weather_Forcast_App/scripts/Crawl_data_from_Vrain_by_API.py


0

In [51]:
# ====== CACH 2: Crawl tu Vrain bang Selenium (can Chrome) ======
# run_script("Weather_Forcast_App/scripts/Crawl_data_from_Vrain_by_Selenium.py")

In [52]:
# ====== CACH 3: Crawl tu OpenWeather + WeatherAPI ======
# run_script("Weather_Forcast_App/scripts/Crawl_data_by_API.py")

---

## 🔀 Phase 2 – Gộp dữ liệu (Data Merging)

Sau khi crawl xong, các file nằm rải rác trong `data/data_crawl/`.  
Bước này sẽ gộp tất cả lại thành **1 dataset thống nhất**.

**Script:** `Merge_xlsx.py`  
**Input:** Tất cả file trong `data/data_crawl/`  
**Output:**
- `data/data_merge/merged_vrain_data.xlsx` – dữ liệu Vrain  
- `data/data_merge/merged_vietnam_weather_data.xlsx` – dữ liệu OpenWeather  

**Cơ chế:**
- Tự động nhận diện file mới chưa merge (dựa vào `merged_files_log.txt`)
- Chuẩn hóa 42 cột theo master schema
- Batch save mỗi 30,000 dòng để tránh mất dữ liệu

In [53]:
# ====== GOP DU LIEU ======
run_script("Weather_Forcast_App/scripts/Merge_xlsx.py")

▶ d:\PROJECT_WEATHER_FORCAST\.venv\Scripts\python.exe Weather_Forcast_App/scripts/Merge_xlsx.py


0

---

## 🧽 Phase 2B – Làm sạch sơ bộ (Initial Cleaning – Cleardata)

**Bước này bắt buộc** trước khi chạy Phase 3.

Sau khi merge, file `merged_vrain_data.xlsx` còn nhiều vấn đề:
- Header bị lặp trong dữ liệu (do nối file)
- Giá trị rác: `"N/A"`, `"null"`, chuỗi rỗng
- Dữ liệu âm ở các cột số
- Cột datetime chưa chuẩn dtype

**Script `Cleardata.py`** là một Django view, không chạy trực tiếp từ CLI được.  
→ Cell bên dưới gọi trực tiếp hàm `perform_cleaning()` để chạy trong notebook.

**Input:** `data/data_merge/merged_vrain_data.xlsx`  
**Output:** `data/data_clean/data_merge_clean/cleaned_merge_merged_vrain_data_*.csv`

In [54]:
# ====== LAM SACH SO BO (Cleardata - perform_cleaning) ======
import sys, os, importlib
sys.path.insert(0, os.getcwd())

import Weather_Forcast_App.scripts.Cleardata as _cd
importlib.reload(_cd)
from Weather_Forcast_App.scripts.Cleardata import perform_cleaning

import pandas as pd

# Doc file merged
merged_file = "data/data_merge/merged_vrain_data.xlsx"
filename = os.path.basename(merged_file)

print(f"Dang doc file: {merged_file}")
df = pd.read_excel(merged_file, engine="openpyxl")
print(f"  Truoc cleaning: {df.shape[0]} dong, {df.shape[1]} cot")

# Chay cleaning
result = perform_cleaning(df, filename, file_type="merged")
print(f"  {result['message']}")
print(f"  Output: {result.get('output_file', 'N/A')}")

Dang doc file: data/data_merge/merged_vrain_data.xlsx
  Truoc cleaning: 13431 dong, 43 cot
Số dòng sau khi loại header: 13431
Mẫu dữ liệu đầu tiên:
    station_id station_name  province district  latitude  longitude  \
0  Bạch Ngọc 1  Bạch Ngọc 1  Hà Giang      NaN       NaN        NaN   
1  Bạch Ngọc 2  Bạch Ngọc 2  Hà Giang      NaN       NaN        NaN   

          timestamp data_source data_quality         data_time  ...  \
0  16/02/2026 09:36         NaN          NaN  16/02/2026 09:36  ...   
1  16/02/2026 09:36         NaN          NaN  16/02/2026 09:36  ...   

   cloud_cover_current  cloud_cover_max  cloud_cover_min  cloud_cover_avg  \
0                  NaN              NaN              NaN              NaN   
1                  NaN              NaN              NaN              NaN   

   visibility_current  visibility_max  visibility_min  visibility_avg  \
0                 NaN             NaN             NaN             NaN   
1                 NaN             NaN         

d:\PROJECT_WEATHER_FORCAST\Weather_Forcast_App\scripts\Cleardata.py:325: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  data_df[col] = pd.to_datetime(data_df[col], errors="coerce")
d:\PROJECT_WEATHER_FORCAST\Weather_Forcast_App\scripts\Cleardata.py:325: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  data_df[col] = pd.to_datetime(data_df[col], errors="coerce")


  Làm sạch dữ liệu hoàn tất
  Output: merged_vrain_data_cleaned_20260307_232127.csv


---

## 🧹 Phase 3 – Làm sạch features (Feature Cleaning)

Dữ liệu sau khi qua Phase 2B đã sạch cơ bản và **cột đã được Cleardata đổi sang English schema** (`rain_total`, `temperature_current`, ...).

| Bước | Chức năng | Ghi chú |
|-------|-----------|---------|
| 1 | Đổi tên cột tiếng Việt → schema chuẩn | ⏩ **SKIP** – Cleardata đã làm |
| 2 | Làm sạch cột mục tiêu `rain_total` (loại NaN/inf) | ✅ Chạy |
| 3 | Chuẩn hóa tên cột thời tiết | ⏩ **SKIP** – Cleardata đã làm |
| 4 | Kiểm tra lần cuối, loại NaN/infinity trong features | ✅ Chạy |

**Input:** File từ Phase 2B (auto-detect)  
**Output:** `...clean_final.csv` – **dataset sẵn sàng cho training**

**Quan trọng:** Cell "Auto-detect" phải chạy trước để xác định file input đúng.

In [55]:
# ====== AUTO-DETECT: Tim file cleaned moi nhat tu Phase 2B ======
import glob

clean_dir = "data/data_clean/data_merge_clean"

# Tim tat ca file cleaned (ca 2 pattern ten)
csv_files = sorted(
    glob.glob(f"{clean_dir}/cleaned_merge_*.csv") +
    glob.glob(f"{clean_dir}/merged_vrain_data_cleaned_*.csv"),
    key=os.path.getmtime, reverse=True
)
# Loai bo file co suffix pipeline (san pham cua Phase 3)
csv_files = [f for f in csv_files
             if ".schema_en" not in f
             and ".clean_target" not in f
             and ".schema_fixed" not in f
             and ".clean_final" not in f]

if csv_files:
    LATEST_CLEAN = csv_files[0]
    CLEAN_BASE = os.path.splitext(LATEST_CLEAN)[0]  # dung cho pipeline chuoi
    print(f"✅ File cleaned moi nhat: {LATEST_CLEAN}")
    print(f"   Base name: {CLEAN_BASE}")
else:
    LATEST_CLEAN = None
    print(f"❌ CHUA CO FILE NAO trong {clean_dir}!")
    print("Hay chay Phase 2B (Cleardata) truoc.")

✅ File cleaned moi nhat: data/data_clean/data_merge_clean\merged_vrain_data_cleaned_20260307_232127.csv
   Base name: data/data_clean/data_merge_clean\merged_vrain_data_cleaned_20260307_232127


In [56]:
# ====== BUOC 1: Doi ten cot theo schema chuan ======
# ⏩ SKIP: Cleardata da chuyen cot sang tieng Anh roi (station_id, rain_total, ...)
# Script nay chi can khi input con dung ten tieng Viet.
print("⏩ SKIP: Cleardata da rename cot sang English schema. Khong can chay buoc nay.")

⏩ SKIP: Cleardata da rename cot sang English schema. Khong can chay buoc nay.


In [57]:
# ====== BUOC 2: Lam sach cot muc tieu (rain_total) ======
import pandas as pd
import numpy as np

TARGET_COL = "rain_total"
SRC_2 = LATEST_CLEAN
DST_2 = CLEAN_BASE + ".clean_target.csv"

print(f"SRC: {SRC_2}")
print(f"DST: {DST_2}")
print(f"Target column: {TARGET_COL}")

df = pd.read_csv(SRC_2)
before = len(df)
df = df[pd.to_numeric(df[TARGET_COL], errors='coerce').notnull()]
df = df[~df[TARGET_COL].isin([np.inf, -np.inf])]
print(f"Số dòng hợp lệ còn lại: {len(df)} (loại {before - len(df)} dòng)")
df.to_csv(DST_2, index=False)
print(f"Đã ghi file: {DST_2}")

SRC: data/data_clean/data_merge_clean\merged_vrain_data_cleaned_20260307_232127.csv
DST: data/data_clean/data_merge_clean\merged_vrain_data_cleaned_20260307_232127.clean_target.csv
Target column: rain_total
Số dòng hợp lệ còn lại: 13431 (loại 0 dòng)
Đã ghi file: data/data_clean/data_merge_clean\merged_vrain_data_cleaned_20260307_232127.clean_target.csv


In [58]:
# ====== BUOC 3: Chuan hoa ten cot thoi tiet ======
# ⏩ SKIP: Cleardata da chuyen cot sang English schema (temperature_current, rain_total, ...)
# Script nay chi can khi cot con tieng Viet (Nhiet do hien tai, ...).
print("⏩ SKIP: Cleardata da rename cot sang English schema. Khong can chay buoc nay.")

⏩ SKIP: Cleardata da rename cot sang English schema. Khong can chay buoc nay.


In [59]:
# ====== BUOC 4: Kiem tra lan cuoi, dam bao data sach ======
import pandas as pd
import numpy as np

TARGET_COL = "rain_total"
SRC_4 = CLEAN_BASE + ".clean_target.csv"
DST_4 = CLEAN_BASE + ".clean_final.csv"

print(f"SRC: {SRC_4}")
print(f"DST: {DST_4}")

df = pd.read_csv(SRC_4)
feature_cols = [col for col in df.columns if col != TARGET_COL]

# Xac dinh cot so (an toan voi StringDtype va extension types)
num_cols = []
for col in feature_cols:
    try:
        if pd.api.types.is_numeric_dtype(df[col]):
            num_cols.append(col)
    except Exception:
        pass

# Loc NaN, inf chi tren cot so
mask = pd.Series(True, index=df.index)
if num_cols:
    mask = mask & (~df[num_cols].isnull().any(axis=1))
    mask = mask & (~df[num_cols].isin([np.inf, -np.inf]).any(axis=1))
    mask = mask & (df[num_cols].abs().le(1e10).all(axis=1))

cleaned = df[mask].copy()
print(f"Số dòng hợp lệ sau khi lọc feature: {len(cleaned)} / {len(df)}")
cleaned.to_csv(DST_4, index=False)
print(f"Đã ghi file: {DST_4} (lọc feature hợp lệ)")

SRC: data/data_clean/data_merge_clean\merged_vrain_data_cleaned_20260307_232127.clean_target.csv
DST: data/data_clean/data_merge_clean\merged_vrain_data_cleaned_20260307_232127.clean_final.csv
Số dòng hợp lệ sau khi lọc feature: 13431 / 13431
Đã ghi file: data/data_clean/data_merge_clean\merged_vrain_data_cleaned_20260307_232127.clean_final.csv (lọc feature hợp lệ)


---

## 🧠 Phase 4 – Huấn luyện mô hình (Model Training)

Dùng Django management command để huấn luyện mô hình Machine Learning.

**Hệ thống hỗ trợ các loại mô hình:**
- `xgboost` – XGBoost (mặc định, nhanh và mạnh)
- `lightgbm` – LightGBM (nhanh hơn, tốt với dataset lớn)
- `catboost` – CatBoost (tốt với dữ liệu có nhiều categorical features)
- `random_forest` – Random Forest (cổ điển, ổn định)
- `two_stage` – Mô hình 2 giai đoạn cho dữ liệu mưa zero-inflated

**Pipeline huấn luyện tự động thực hiện:**
1. Load config từ file JSON
2. Load dữ liệu + validate schema
3. Chia train / validation / test
4. Feature engineering (lag, rolling, diff, cyclical encoding...)
5. Loại bỏ features nhiễu / hằng số
6. Huấn luyện mô hình + early stopping
7. Đánh giá (RMSE, MAE, R², MBE, Pearson, Accuracy, F1, CSI)
8. Lưu artifacts (Model.pkl, Pipeline.pkl, Metrics.json...)

**Config:**
- **Mặc định** (không cần `--config`): `Weather_Forcast_App/Machine_learning_model/config/train_config.json`
- **Tuỳ chọn** (dùng `--config path`): các file trong thư mục `config/` ở project root:

```
config/train_config.json               ← config gốc (trỏ data cũ, ít dùng)
config/train_config_v6.json             ← phiên bản mới nhất
config/train_config_optuna_best.json    ← config tối ưu từ Optuna
config/train_config_optuna_v2_best.json ← config tối ưu v2
config/train_config_heavy_rain_v3.json  ← chuyên cho mưa lớn
```

> ℹ️ Cell "Kiểm tra config" bên dưới sẽ xác nhận file data có tồn tại trước khi train.

**Artifacts được lưu tại:** `Weather_Forcast_App/Machine_learning_artifacts/latest/`

In [60]:
# ====== KIEM TRA CONFIG TRUOC KHI TRAIN ======
import json

# Config chinh cua ML module (dung boi manage.py train_model)
ml_config = "Weather_Forcast_App/Machine_learning_model/config/train_config.json"
with open(ml_config, "r", encoding="utf-8") as f:
    cfg = json.load(f)

data_cfg = cfg.get("data", {})
data_file = f"data/data_clean/data_merge_clean/{data_cfg.get('filename', 'N/A')}"
exists = os.path.exists(data_file)

print(f"ML Config:     {ml_config}")
print(f"Target column: {cfg.get('target_column', 'N/A')}")
print(f"Train data:    {data_file}")
print(f"File ton tai:  {'✅ CO' if exists else '❌ KHONG — can sua lai filename trong config!'}")

ML Config:     Weather_Forcast_App/Machine_learning_model/config/train_config.json
Target column: rain_total
Train data:    data/data_clean/data_merge_clean/merged_vrain_data_cleaned_20260307_223506.clean_final.csv
File ton tai:  ✅ CO


In [61]:
# ====== HUAN LUYEN MO HINH (dung config mac dinh) ======
run_script("manage.py", "train_model")

▶ d:\PROJECT_WEATHER_FORCAST\.venv\Scripts\python.exe manage.py train_model


0

In [62]:
# ====== HUAN LUYEN VOI CONFIG TUY CHON ======
# Uncomment 1 dong ben duoi theo nhu cau:

# run_script("manage.py", "train_model", "--config", "config/train_config_v6.json")
# run_script("manage.py", "train_model", "--config", "config/train_config_optuna_best.json")
# run_script("manage.py", "train_model", "--config", "config/train_config_heavy_rain_v3.json")

In [63]:
# ====== DRY-RUN: Kiem tra config truoc khi train (khong train that) ======
# run_script("manage.py", "train_model", "--config", "config/train_config.json", "--dry-run")

---

## ⚙️ Phase 4B (Optional) – Tối ưu siêu tham số (Hyperparameter Tuning)

**Không bắt buộc** – chỉ chạy khi muốn tìm bộ tham số tốt nhất cho mô hình.

Sử dụng **Optuna** (TPE Sampler + Pruning) để tự động:
- Thử hàng trăm bộ tham số khác nhau
- Dừng sớm nếu trial không triển vọng (pruning)
- Tự động lưu config tốt nhất vào `config/train_config_optuna_*_best.json`

Sau khi tune xong, dùng config tối ưu để train lại ở Phase 4.

In [64]:
# ====== OPTUNA TUNING (mat nhieu thoi gian, chi chay khi can) ======
# run_script("scripts/optuna_twostage_v2.py")

---

## 🔮 Phase 5 – Dự báo (Forecast / Prediction)

Sau khi train xong, mô hình đã được lưu dưới dạng **artifacts** gồm:
- `Model.pkl` – mô hình đã train
- `Transform_pipeline.pkl` – pipeline xử lí features
- `Feature_list.json` – danh sách features đã dùng
- `Metrics.json` – kết quả đánh giá

**Có 2 cách chạy dự báo:**

### Cách 1: Chạy script tự động (WeatherForcast.py)
- Load artifacts → Build features → Predict → Xuất `predictions.csv`
- Tự động lọc bỏ các dòng lỗi từ `debug_top50_errors.csv`

### Cách 2: Gọi API qua Python code
- Linh hoạt hơn, có thể predict trên bất kỳ DataFrame nào

In [ ]:
# ====== CACH 1: Chay script du bao tu dong ======
# Input: file cleaned cuoi cung tu Phase 3
forecast_input = DST_4  # file clean_final.csv
forecast_output = "data/data_forecast/forecast_results.csv"
print(f"Input:  {forecast_input}")
print(f"Output: {forecast_output}")
run_script(
    "Weather_Forcast_App/Machine_learning_model/WeatherForcast/WeatherForcast.py",
    "-i", forecast_input,
    "-o", forecast_output
)

Input:  data/data_clean/data_merge_clean\merged_vrain_data_cleaned_20260307_232127.clean_final.csv
Output: forecast_results.csv
▶ d:\PROJECT_WEATHER_FORCAST\.venv\Scripts\python.exe Weather_Forcast_App/Machine_learning_model/WeatherForcast/WeatherForcast.py -i data/data_clean/data_merge_clean\merged_vrain_data_cleaned_20260307_232127.clean_final.csv -o forecast_results.csv


0

In [66]:
# ====== CACH 2: Goi API predict bang Python code ======
# import pandas as pd
# from Weather_Forcast_App.Machine_learning_model.interface.predictor import WeatherPredictor
#
# # Load mo hinh da train
# predictor = WeatherPredictor.from_artifacts(
#     "Weather_Forcast_App/Machine_learning_artifacts/latest"
# )
#
# # Doc du lieu moi can du bao
# df_new = pd.read_csv("data/data_crawl/Bao_cao_20260301_091925.csv")
#
# # Chay du bao
# result = predictor.predict(df_new)
# print("Du bao luong mua:")
# print(result["predictions"])

---

## 📊 Phase 6 (Bonus) – Chẩn đoán lỗi mô hình (Diagnostics)

Sau khi train và predict, có thể chạy **diagnostics** để phân tích các dòng dữ liệu mà mô hình dự đoán sai nhiều nhất.  
Kết quả xuất ra `debug_top50_errors.csv` – file này sẽ được dùng lại trong lần train tiếp theo để lọc bỏ các dòng nhiễu.

In [67]:
# ====== CHAN DOAN LOI MO HINH ======
# run_script("scripts/run_diagnostics.py")

---

## 🏁 Tổng kết

| Phase | Mô tả | Thời gian ước tính |
|-------|---------|-------------------|
| **1. Crawl** | Thu thập dữ liệu từ Vrain / OpenWeather | 5–30 phút |
| **2. Merge** | Gộp tất cả file lại | 1–5 phút |
| **3. Clean** | Làm sạch 5 bước | 2–10 phút |
| **4. Train** | Huấn luyện mô hình ML | 5–60 phút |
| **4B. Tune** | Tối ưu siêu tham số (optional) | 1–24 giờ |
| **5. Forecast** | Dự báo thời tiết | 1–5 phút |

> **Tip:** Nếu chỉ muốn test nhanh, bỏ qua Phase 4B (Tuning) và dùng config mặc định.

---

© 2026 Weather Forecast System – *Trembling Process Notebook*